In [ ]:
import os
import argparse
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from PIL import Image
from diffusers import (
    StableDiffusionPipeline,
    UNet2DConditionModel,
    AutoencoderKL,
    DDPMScheduler,
)
from diffusers.optimization import get_scheduler
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator

In [ ]:
class DreamBoothDataset(Dataset):
    def __init__(self, captions_file: str, image_size: int = 512):
        self.items = []
        with open(captions_file, encoding="utf-8") as f:
            for line in f:
                path, caption = line.strip().split("|")
                self.items.append((path, caption))
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, caption = self.items[idx]
        image = Image.open(path).convert("RGB")
        return self.transform(image), caption

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="DreamBooth fine-tuning")
    parser.add_argument("--model_id",    type=str,   default="runwayml/stable-diffusion-v1-5")
    parser.add_argument("--train_captions", type=str, required=True)
    parser.add_argument("--val_captions",   type=str, default=None)
    parser.add_argument("--output_dir",  type=str,   default="./dreambooth-finetuned")
    parser.add_argument("--cache_dir",   type=str,   default="./cache")
    parser.add_argument("--batch_size",  type=int,   default=1)
    parser.add_argument("--image_size",  type=int,   default=512)
    parser.add_argument("--lr",          type=float, default=5e-6)
    parser.add_argument("--num_epochs",  type=int,   default=3)
    parser.add_argument("--warmup_steps", type=int,  default=0)
    parser.add_argument("--mixed_precision", choices=["no","fp16","bf16"], default="fp16")
    parser.add_argument("--grad_accum_steps", type=int, default=1)
    parser.add_argument("--log_dir",     type=str,   default="./logs")
    parser.add_argument("--num_val_samples", type=int, default=5)
    return parser.parse_args()

In [ ]:
def load_models(args, device):
    tokenizer = CLIPTokenizer.from_pretrained(args.model_id, cache_dir=args.cache_dir)
    text_encoder = CLIPTextModel.from_pretrained(args.model_id, cache_dir=args.cache_dir).to(device)
    vae = AutoencoderKL.from_pretrained(args.model_id, subfolder="vae", cache_dir=args.cache_dir).to(device)
    unet = UNet2DConditionModel.from_pretrained(args.model_id, subfolder="unet", cache_dir=args.cache_dir).to(device)
    noise_scheduler = DDPMScheduler.from_pretrained(args.model_id, subfolder="scheduler", cache_dir=args.cache_dir)
    return tokenizer, text_encoder, vae, unet, noise_scheduler

In [ ]:
def build_dataloaders(args):
    train_ds = DreamBoothDataset(args.train_captions, image_size=args.image_size)
    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        shuffle=True,
        drop_last=True
    )
    val_loader = None
    if args.val_captions:
        val_ds = DreamBoothDataset(args.val_captions, image_size=args.image_size)
        val_loader = DataLoader(
            val_ds,
            batch_size=args.batch_size,
            shuffle=False,
            drop_last=False
        )
    return train_loader, val_loader

In [ ]:
def configure_optimizers(unet, args, train_loader):
    optimizer = torch.optim.AdamW(unet.parameters(), lr=args.lr)
    total_steps = args.num_epochs * len(train_loader) // args.grad_accum_steps
    lr_scheduler = get_scheduler(
        name="cosine",
        optimizer=optimizer,
        num_warmup_steps=args.warmup_steps,
        num_training_steps=total_steps,
    )
    return optimizer, lr_scheduler

In [ ]:
def train_one_epoch(unet, vae, noise_scheduler, tokenizer,
                    train_loader, optimizer, lr_scheduler,
                    accelerator, epoch, writer, args):
    device = accelerator.device
    unet.train()
    global_step = 0

    for step, (images, captions) in enumerate(train_loader, start=1):
        # Токенизация
        inputs = tokenizer(
            captions,
            padding="max_length",
            truncation=True,
            max_length=tokenizer.model_max_length,
            return_tensors="pt"
        ).to(device)

        # Латенты
        latents = vae.encode(images.to(device)).latent_dist.sample() * 0.18215
        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0,
            noise_scheduler.num_train_timesteps,
            (latents.shape[0],),
            device=device
        )
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Предсказание шума и loss
        noise_pred = unet(noisy_latents, timesteps,
                          encoder_hidden_states=inputs.input_ids).sample
        loss = torch.nn.functional.mse_loss(noise_pred, noise)
        loss = loss / args.grad_accum_steps

        accelerator.backward(loss)
        if step % args.grad_accum_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if writer:
                writer.add_scalar("train/loss", loss.item() * args.grad_accum_steps, global_step)
                writer.add_scalar("train/lr", lr_scheduler.get_last_lr()[0], global_step)

    print(f"Epoch {epoch} training completed.")

In [ ]:
def validate_and_generate(text_encoder, vae, unet, tokenizer,
                          noise_scheduler, val_loader, accelerator,
                          args, epoch):
    if not val_loader or not accelerator.is_main_process:
        return

    device = accelerator.device
    pipeline = StableDiffusionPipeline(
        text_encoder=text_encoder,
        vae=vae,
        unet=unet,
        tokenizer=tokenizer,
        scheduler=noise_scheduler,
        safety_checker=None,
        feature_extractor=None,
    ).to(device)
    pipeline.enable_attention_slicing()

    sample_dir = Path(args.output_dir) / f"samples_epoch_{epoch}"
    sample_dir.mkdir(parents=True, exist_ok=True)

    for i, (_, captions) in enumerate(val_loader):
        prompt = captions[0]
        with torch.autocast(device.type):
            image = pipeline(prompt, num_inference_steps=50, guidance_scale=7.5).images[0]
        image.save(sample_dir / f"sample_{i+1}.png")
        if i + 1 >= args.num_val_samples:
            break

    print(f"Epoch {epoch} validation samples saved.")

In [ ]:
def save_checkpoint(unet, text_encoder, vae, tokenizer, args, epoch):
    ckpt_dir = Path(args.output_dir) / f"checkpoint_epoch_{epoch}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    unet.save_pretrained(ckpt_dir / "unet")
    text_encoder.save_pretrained(ckpt_dir / "text_encoder")
    vae.save_pretrained(ckpt_dir / "vae")
    tokenizer.save_pretrained(ckpt_dir / "tokenizer")
    print(f"Checkpoint for epoch {epoch} saved.")

In [ ]:
def save_final_model(text_encoder, vae, unet, tokenizer, noise_scheduler, args):
    pipeline = StableDiffusionPipeline(
        text_encoder=text_encoder,
        vae=vae,
        unet=unet,
        tokenizer=tokenizer,
        scheduler=noise_scheduler,
    )
    pipeline.save_pretrained(args.output_dir)
    print(f"Final model saved in {args.output_dir}")

In [ ]:
args = parse_args()
os.environ["HF_HOME"] = args.cache_dir

In [ ]:
accelerator = Accelerator(mixed_precision=args.mixed_precision)
device = accelerator.device

In [ ]:
tokenizer, text_encoder, vae, unet, noise_scheduler = load_models(args, device)
train_loader, val_loader = build_dataloaders(args)
optimizer, lr_scheduler = configure_optimizers(unet, args, train_loader)

In [ ]:
unet, optimizer, train_loader, lr_scheduler = accelerator.prepare(
    unet, optimizer, train_loader, lr_scheduler
)

In [ ]:
writer = SummaryWriter(log_dir=args.log_dir) if accelerator.is_main_process else None

In [ ]:
for epoch in range(1, args.num_epochs + 1):
    train_one_epoch(
        unet, vae, noise_scheduler, tokenizer,
        train_loader, optimizer, lr_scheduler,
        accelerator, epoch, writer, args
    )

    validate_and_generate(
        text_encoder, vae, unet, tokenizer,
        noise_scheduler, val_loader, accelerator,
        args, epoch
    )

    if accelerator.is_main_process:
        save_checkpoint(unet, text_encoder, vae, tokenizer, args, epoch)
    
    if accelerator.is_main_process:
        save_final_model(text_encoder, vae, unet, tokenizer, noise_scheduler, args)
        if writer:
            writer.close()